# Demonstração de uso do MariaDB

Simula um **client** comunicando com o banco de dados relacional pela rede.

In [16]:
import os

import geopandas as gpd
import ipywidgets as widgets
import leafmap
import pandas as pd
from dotenv import load_dotenv
from ipyleaflet import WidgetControl
from shapely.geometry import shape
from sqlalchemy import create_engine, text


In [17]:
load_dotenv()

engine = create_engine(
    "mysql+pymysql://{user}:{pswd}@{host}:{port}/{db}".format(
        user=os.getenv("MARIADB_USER"),
        pswd=os.getenv("MARIADB_PASSWORD"),
        host=os.getenv("HOST"),
        port=os.getenv("PORT"),
        db=os.getenv("MARIADB_DATABASE"),
    )
)


## Consulta espacial com JOIN dinâmico

Desenhe um polígono ou retângulo no mapa abaixo e clique em **Consultar área** para filtrar e unir as bases na sequência `focos_de_fogo` → `municipios` → `estados` pelas suas chaves estrangeiras (`municipio_id`, `estado_id`), usando `ST_Intersects` sobre as geometrias. Os focos encontrados aparecem diretamente no mapa — clique em um ponto para ver os atributos de `municipio`/`estado` trazidos pelo JOIN.

In [19]:
def query_focos_in_roi(m: leafmap.Map) -> gpd.GeoDataFrame:
    if m.user_roi is None:
        raise ValueError("Desenhe uma área no mapa antes de consultar.")

    roi_wkt = shape(m.user_roi["geometry"]).wkt

    query = text(
        """
        SELECT
            f.id_foco_bdq,
            f.satelite,
            f.data_hora_gmt,
            f.frp,
            f.risco_fogo,
            f.bioma,
            m.nome AS municipio,
            e.nome AS estado,
            e.regiao,
            ST_AsBinary(f.geometry) AS geometry
        FROM focos_de_fogo f
        JOIN municipios m ON m.id = f.municipio_id
        JOIN estados e ON e.id = m.estado_id
        WHERE ST_Intersects(f.geometry, ST_GeomFromText(:roi_wkt, 4674))
        """
    )

    with engine.connect() as conn:
        df = pd.read_sql(query, con=conn, params={"roi_wkt": roi_wkt})

    return gpd.GeoDataFrame(
        df.drop(columns=["geometry"]),
        geometry=gpd.GeoSeries.from_wkb(df["geometry"]),
        crs="EPSG:4674",
    )


m = leafmap.Map(center=[-14, -52], zoom=4)
m.add_basemap("OpenStreetMap")

status = widgets.HTML("Desenhe uma área e clique em <b>Consultar área</b>.")
button = widgets.Button(description="Consultar área", button_style="primary", icon="search")


def on_click(_):
    try:
        focos_in_roi = query_focos_in_roi(m)
    except ValueError as error:
        status.value = f"⚠️ {error}"
        return

    for layer in list(m.layers):
        if layer.name == "Focos de fogo":
            m.remove_layer(layer)

    m.add_gdf(focos_in_roi.to_crs("EPSG:4326"), layer_name="Focos de fogo", info_mode="on_click")
    status.value = f"{len(focos_in_roi)} focos de fogo encontrados nesta área."


button.on_click(on_click)
m.add_control(WidgetControl(widget=button, position="topright"))
m.add_control(WidgetControl(widget=status, position="bottomleft"))
m


OpenStreetMap has been already added before.


Map(center=[-14, -52], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…